# Google Fit Takeout — Exploratory Analysis

This notebook explores the Google Fit export in `Takeout/Fit/`. The export has four parts:

| Folder | Contents | Granularity |
|---|---|---|
| `Métricas de actividad diaria/` | one CSV per day + one aggregated CSV | 15-minute buckets / daily |
| `Todas las sesiones/` | one JSON per tracked session | one row per workout |
| `Actividades/` | one TCX per tracked session | trackpoint (~20–60 s) |
| `Todos los datos/` | raw & derived Fit data streams | raw sample |

Sections:

1. [Export inventory](#1) — what is actually in the box
2. [Daily metrics](#2) — steps, distance, calories, cardio points over ~3 years
3. [Intraday rhythm](#3) — 15-minute buckets: what a day looks like
4. [Sessions](#4) — walking / running / biking workouts
5. [TCX trackpoints](#5) — inside a single session
6. [Raw data streams](#6) — `Todos los datos/`, including what is empty
7. [Takeaways](#7)

## Setup

In [ ]:
import json
import re
import time
import unicodedata
import xml.etree.ElementTree as ET
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', lambda v: f'{v:,.2f}')

# Locate Takeout/Fit from wherever the notebook is opened.
def find_fit_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for cand in [start, *start.parents]:
        hit = cand / 'Takeout' / 'Fit'
        if hit.is_dir():
            return hit
    raise FileNotFoundError('Takeout/Fit not found above ' + str(start))

FIT = find_fit_root()

def _norm(text):
    '''Lowercase and strip accents so Spanish/English exports both match.'''
    text = unicodedata.normalize('NFKD', str(text))
    return ''.join(c for c in text if not unicodedata.combining(c)).strip().lower()

def _subdir(*keywords):
    for d in sorted(p for p in FIT.iterdir() if p.is_dir()):
        n = _norm(d.name)
        if any(k in n for k in keywords):
            return d
    raise FileNotFoundError('no subfolder matching ' + repr(keywords))

DAILY_DIR      = _subdir('metricas', 'daily activity metrics')
SESSIONS_DIR   = _subdir('sesiones', 'all sessions')
ACTIVITIES_DIR = _subdir('actividades', 'activities')
ALLDATA_DIR    = _subdir('todos los datos', 'all data')

TZ = 'Europe/Madrid'   # local timezone of the export (offsets in filenames: +01:00 / +02:00)

print('Fit root:', FIT)
for d in (DAILY_DIR, SESSIONS_DIR, ACTIVITIES_DIR, ALLDATA_DIR):
    print(f'  {d.name:<32} {len(list(d.iterdir())):>5} files')

### Plot style

One place for colors so every chart in the notebook reads as one system:
a fixed categorical order (blue → orange → aqua, never cycled), a single-hue
blue ramp for magnitude, and recessive grid/axis ink.

In [ ]:
SURFACE   = '#fcfcfb'
INK       = '#0b0b0b'
INK_SOFT  = '#52514e'
INK_MUTED = '#898781'
GRID      = '#e1e0d9'
AXIS      = '#c3c2b7'

# Categorical slots, assigned in fixed order (never cycled, never by rank).
SERIES = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300', '#4a3aa7', '#e34948']
# Stable color per activity type -- follows the entity, not its rank in a chart.
ACT_COLOR = {'walking': SERIES[0], 'running': SERIES[1], 'biking': SERIES[2]}

# Sequential ramp (one hue, light -> dark) for heatmaps.
BLUES = LinearSegmentedColormap.from_list(
    'fit_blue',
    ['#eef5fd', '#cde2fb', '#9ec5f4', '#6da7ec', '#3987e5', '#256abf', '#184f95', '#0d366b'],
)
BLUES.set_bad(SURFACE)

mpl.rcParams.update({
    'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE, 'savefig.facecolor': SURFACE,
    'figure.dpi': 110, 'figure.autolayout': False,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Segoe UI', 'DejaVu Sans', 'sans-serif'],
    'font.size': 9,
    'axes.edgecolor': AXIS, 'axes.labelcolor': INK_SOFT, 'axes.titlecolor': INK,
    'axes.titlesize': 11, 'axes.titleweight': 'semibold', 'axes.titlelocation': 'left',
    'axes.titlepad': 10, 'axes.labelsize': 9,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.8,
    'xtick.color': INK_MUTED, 'ytick.color': INK_MUTED,
    'xtick.labelcolor': INK_SOFT, 'ytick.labelcolor': INK_SOFT,
    'xtick.labelsize': 8, 'ytick.labelsize': 8,
    'lines.linewidth': 2, 'lines.markersize': 5,
    'legend.frameon': False, 'legend.fontsize': 8.5, 'legend.labelcolor': INK_SOFT,
})

def finish(ax, title=None, subtitle=None, ylabel=None, xlabel=None, yfmt=True):
    '''Common chart chrome: title/subtitle, horizontal-only grid, thousands ticks.'''
    if title:
        ax.set_title(title, pad=22 if subtitle else 10)
    if subtitle:
        # Offset in points, not axes fraction: short axes would otherwise
        # push the subtitle into the title.
        ax.annotate(subtitle, xy=(0, 1), xycoords='axes fraction', xytext=(0, 7),
                    textcoords='offset points', fontsize=8.5, color=INK_MUTED, va='bottom')
    ax.set_ylabel(ylabel or '', color=INK_SOFT)
    ax.set_xlabel(xlabel or '', color=INK_SOFT)
    ax.grid(axis='x', visible=False)
    ax.grid(axis='y', visible=True)
    ax.set_axisbelow(True)
    if yfmt:
        ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda v, _: f'{v:,.0f}'))
    return ax

<a id="1"></a>
## 1. Export inventory

Before analysing anything, measure the box: how many files, how big, and which
date range each part covers. Session and activity filenames carry the local
timestamp, so the coverage window can be read without opening them.

In [ ]:
def dir_stats(d, pattern='*'):
    files = [p for p in d.glob(pattern) if p.is_file()]
    size = sum(p.stat().st_size for p in files)
    dates = sorted(m.group(0) for m in (re.match(r'\d{4}-\d{2}-\d{2}', p.name) for p in files) if m)
    return {
        'folder': d.name,
        'files': len(files),
        'size_mb': round(size / 1e6, 1),
        'first': dates[0] if dates else '',
        'last': dates[-1] if dates else '',
    }

inventory = pd.DataFrame([
    dir_stats(DAILY_DIR, '*.csv'),
    dir_stats(SESSIONS_DIR, '*.json'),
    dir_stats(ACTIVITIES_DIR, '*.tcx'),
    dir_stats(ALLDATA_DIR, '*.json'),
])
inventory

Two different coverage windows to keep in mind: the aggregated daily metrics
reach back to late 2023, while per-session files (JSON/TCX) only start in 2025 —
they come from a later phone. Anything session-based below is therefore a
narrower window than the daily series.

<a id="2"></a>
## 2. Daily metrics

`Métricas de actividad diaria/Métricas de actividad diaria.csv` is the pre-aggregated
one-row-per-day table. Column names are Spanish; they are normalised to snake_case here.

In [ ]:
COLMAP = {
    'fecha': 'date',
    'hora de inicio': 'start',
    'hora de finalizacion': 'end',
    'recuento de minutos activos': 'active_minutes',
    'calorias (kcal)': 'calories',
    'distancia (m)': 'distance_m',
    'puntos cardio': 'cardio_points',
    'minutos de cardio': 'cardio_minutes',
    'velocidad media (m/s)': 'speed_mean_ms',
    'velocidad maxima (m/s)': 'speed_max_ms',
    'velocidad minima (m/s)': 'speed_min_ms',
    'recuento de pasos': 'steps',
    'peso medio (kg)': 'weight_mean_kg',
    'peso maximo (kg)': 'weight_max_kg',
    'peso minimo (kg)': 'weight_min_kg',
    'bicicleta duracion (ms)': 'biking_ms',
    'caminar duracion (ms)': 'walking_ms',
    'correr duracion (ms)': 'running_ms',
}

def tidy_columns(df):
    return df.rename(columns={c: COLMAP.get(_norm(c), _norm(c).replace(' ', '_')) for c in df.columns})

agg_csv = DAILY_DIR / f'{DAILY_DIR.name}.csv'
daily = tidy_columns(pd.read_csv(agg_csv, encoding='utf-8'))
daily['date'] = pd.to_datetime(daily['date'])
daily = daily.sort_values('date').reset_index(drop=True)

# Derived features used throughout.
daily['distance_km'] = daily['distance_m'] / 1000
for act in ('walking', 'running', 'biking'):
    daily[f'{act}_min'] = daily[f'{act}_ms'] / 60000
daily['year'] = daily['date'].dt.year
daily['month'] = daily['date'].dt.to_period('M').dt.to_timestamp()
daily['weekday'] = daily['date'].dt.dayofweek          # 0 = Monday
daily['is_weekend'] = daily['weekday'] >= 5

print(f'{len(daily)} days, {daily.date.min():%Y-%m-%d} -> {daily.date.max():%Y-%m-%d}')
gap = pd.date_range(daily.date.min(), daily.date.max()).difference(daily.date)
print(f'calendar days with no row: {len(gap)}')
daily.head()

### How complete is each column?

In [ ]:
coverage = pd.DataFrame({
    'non_null': daily.notna().sum(),
    'pct': (daily.notna().mean() * 100).round(1),
}).loc[['steps', 'distance_km', 'active_minutes', 'cardio_points', 'cardio_minutes',
        'calories', 'speed_mean_ms', 'walking_min', 'running_min', 'biking_min',
        'weight_mean_kg']]
coverage

Read this before trusting any average:

- **steps / distance / speed** are complete — these are the reliable series.
- **calories** only exist for ~60% of days (a second app, Pacer, fed them for part of the period),
  so a "calories per year" comparison would compare different measurement regimes.
- **walking/running/biking durations** are only filled on days with a *tracked session*,
  not on every day with movement.
- **weight** has exactly one measurement in the whole export — there is no weight series to plot.

In [ ]:
summary = daily[['steps', 'distance_km', 'active_minutes', 'cardio_points', 'calories']].describe().T
summary['median'] = daily[summary.index].median()
summary[['count', 'mean', 'median', 'std', 'min', '25%', '75%', 'max']]

### Steps over time

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(daily.date, daily.steps, color=SERIES[0], lw=0.7, alpha=0.28)
ax.plot(daily.date, daily.steps.rolling(7, min_periods=3).mean(),
        color=SERIES[0], lw=1.4, alpha=0.75, label='7-day average')
ax.plot(daily.date, daily.steps.rolling(30, min_periods=10).mean(),
        color=SERIES[1], lw=2.4, label='30-day average')
ax.axhline(10000, color=INK_MUTED, lw=1, ls=(0, (4, 3)))
ax.annotate('10,000 steps', xy=(daily.date.iloc[5], 10000), xytext=(0, 5),
            textcoords='offset points', fontsize=8, color=INK_MUTED)
ax.legend(loc='upper left', ncols=2)
finish(ax, 'Daily step count',
       f'{daily.date.min():%b %Y} – {daily.date.max():%b %Y} · thin line = raw days',
       ylabel='steps')
plt.show()

In [ ]:
metrics = [('distance_km', 'Distance (km)'), ('active_minutes', 'Active minutes'),
           ('cardio_points', 'Cardio points')]
fig, axes = plt.subplots(3, 1, figsize=(11, 7.5), sharex=True)
for ax, (col, label) in zip(axes, metrics):
    ax.plot(daily.date, daily[col], color=SERIES[0], lw=0.6, alpha=0.22)
    ax.plot(daily.date, daily[col].rolling(30, min_periods=10).mean(), color=SERIES[0], lw=2.2)
    finish(ax, label if ax is axes[0] else None, ylabel=label)
    ax.set_title(label, fontsize=10, pad=6)
axes[0].set_title('Distance, active minutes and cardio points — 30-day average', pad=22)
axes[0].annotate('each panel keeps its own scale · thin line = raw days', xy=(0, 1),
                 xycoords='axes fraction', xytext=(0, 7), textcoords='offset points',
                 fontsize=8.5, color=INK_MUTED, va='bottom')
plt.tight_layout()
plt.show()

### Weekly and monthly rhythm

In [ ]:
DAY_NAMES = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4), gridspec_kw={'width_ratios': [1, 1.25]})

by_day = [daily.loc[daily.weekday == d, 'steps'].dropna() for d in range(7)]
bp = ax1.boxplot(by_day, patch_artist=True, widths=0.6, showfliers=False,
                 medianprops=dict(color=SURFACE, lw=2),
                 whiskerprops=dict(color=AXIS), capprops=dict(color=AXIS))
for i, box in enumerate(bp['boxes']):
    box.set(facecolor=SERIES[2] if i >= 5 else SERIES[0], edgecolor=SURFACE, lw=2, alpha=0.9)
ax1.set_xticks(range(1, 8), DAY_NAMES)
finish(ax1, 'Steps by weekday', 'box = middle 50% of days · weekend in aqua', ylabel='steps')

month_avg = daily.groupby(daily.date.dt.month)['steps'].mean()
ax2.bar(month_avg.index, month_avg.values, color=SERIES[0], width=0.72)
ax2.set_xticks(range(1, 13), ['J', 'F', 'M', 'A', 'M', 'J', 'J', 'A', 'S', 'O', 'N', 'D'])
finish(ax2, 'Steps by month of year', 'all years pooled', ylabel='mean steps')
plt.tight_layout()
plt.show()

print(daily.groupby('is_weekend')['steps'].agg(['mean', 'median', 'count']).rename(
    index={False: 'weekday', True: 'weekend'}))

### Calendar view

In [ ]:
def calendar_heatmap(df, value='steps', cmap=BLUES, title=None, subtitle=None):
    '''One row-block per year: weekday x ISO week grid of a daily value.'''
    years = sorted(df.year.unique())
    fig, axes = plt.subplots(len(years), 1, figsize=(11, 1.55 * len(years) + 0.9))
    axes = np.atleast_1d(axes)
    vmax = np.nanpercentile(df[value], 98)
    for ax, yr in zip(axes, years):
        sub = df[df.year == yr]
        grid = np.full((7, 54), np.nan)
        week = ((sub.date.dt.dayofyear + pd.Timestamp(year=yr, month=1, day=1).dayofweek - 1) // 7).values
        grid[sub.weekday.values, week] = sub[value].values
        ax.imshow(np.ma.masked_invalid(grid), aspect='auto', cmap=cmap, vmin=0, vmax=vmax,
                  interpolation='nearest')
        ax.set_yticks(range(7), DAY_NAMES, fontsize=7)
        starts = [((pd.Timestamp(year=yr, month=m, day=1).dayofyear
                    + pd.Timestamp(year=yr, month=1, day=1).dayofweek - 1) // 7) for m in range(1, 13)]
        ax.set_xticks(starts, ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'], fontsize=7.5)
        ax.grid(False)
        for s in ax.spines.values():
            s.set_visible(False)
        ax.set_ylabel(str(yr), rotation=0, ha='right', va='center', labelpad=14,
                      fontsize=10, color=INK)
    axes[0].set_title(title or f'{value} by day', pad=24)
    axes[0].annotate(subtitle or '', xy=(0, 1), xycoords='axes fraction', xytext=(0, 8),
                     textcoords='offset points', fontsize=8.5, color=INK_MUTED, va='bottom')
    cb = fig.colorbar(axes[0].images[0], ax=axes, fraction=0.02, pad=0.015)
    cb.outline.set_visible(False)
    cb.ax.tick_params(labelsize=7.5, color=AXIS)
    return fig

calendar_heatmap(daily, 'steps',
                 title='Steps, day by day',
                 subtitle='darker = more steps · blank = no data · scale clipped at the 98th percentile')
plt.show()

### Hitting the 10,000-step goal

In [ ]:
goal = daily.dropna(subset=['steps']).copy()
goal['hit'] = goal.steps >= 10000
monthly_rate = goal.groupby('month')['hit'].agg(['mean', 'size'])
monthly_rate = monthly_rate[monthly_rate['size'] >= 20]   # drop partial months

fig, ax = plt.subplots(figsize=(11, 3.8))
ax.bar(monthly_rate.index, monthly_rate['mean'] * 100, width=22, color=SERIES[0])
overall = goal.hit.mean() * 100
ax.axhline(overall, color=SERIES[1], lw=2)
ax.annotate(f'overall {overall:.0f}%', xy=(monthly_rate.index[-1], overall),
            xytext=(6, 4), textcoords='offset points', fontsize=8.5, color=SERIES[1])
ax.set_ylim(0, 100)
finish(ax, 'Share of days reaching 10,000 steps',
       'months with at least 20 days of data', ylabel='% of days')
plt.show()

# Longest run of consecutive goal days.
streak_id = (~goal.hit).cumsum()
streaks = goal[goal.hit].groupby(streak_id).agg(days=('hit', 'size'),
                                                start=('date', 'min'), end=('date', 'max'))
print('Longest 10k streaks:')
print(streaks.sort_values('days', ascending=False).head(5).to_string(index=False))

### Time spent per tracked activity

In [ ]:
act_month = daily.groupby('month')[['walking_min', 'running_min', 'biking_min']].sum()
act_month = act_month[act_month.sum(axis=1) > 0]

fig, ax = plt.subplots(figsize=(11, 4))
bottom = np.zeros(len(act_month))
for col, name in [('walking_min', 'Walking'), ('running_min', 'Running'), ('biking_min', 'Biking')]:
    ax.bar(act_month.index, act_month[col], bottom=bottom, width=22,
           color=ACT_COLOR[name.lower()], label=name, edgecolor=SURFACE, lw=1)
    bottom += act_month[col].values
ax.legend(loc='upper left', ncols=3)
finish(ax, 'Tracked activity minutes per month',
       'only days with a recorded session contribute', ylabel='minutes')
plt.show()

print((daily[['walking_min', 'running_min', 'biking_min']].sum() / 60).round(1).rename('total hours'))

### How the daily metrics relate to each other

In [ ]:
cols = ['steps', 'distance_km', 'active_minutes', 'cardio_points', 'cardio_minutes',
        'calories', 'speed_mean_ms', 'speed_max_ms']
corr = daily[cols].corr()

fig, ax = plt.subplots(figsize=(6.6, 5.4))
im = ax.imshow(corr, cmap=LinearSegmentedColormap.from_list(
    'div', ['#184f95', '#9ec5f4', '#f0efec', '#f0a0a0', '#c02b2b']), vmin=-1, vmax=1)
ax.set_xticks(range(len(cols)), cols, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(cols)), cols, fontsize=8)
ax.grid(False)
for s in ax.spines.values():
    s.set_visible(False)
for i in range(len(cols)):
    for j in range(len(cols)):
        v = corr.iloc[i, j]
        ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=7.5,
                color='#ffffff' if abs(v) > 0.62 else INK_SOFT)
cb = fig.colorbar(im, ax=ax, fraction=0.045, pad=0.03)
cb.outline.set_visible(False)
ax.set_title('Correlation between daily metrics', pad=22)
ax.annotate('Pearson r · blue = negative, red = positive', xy=(0, 1),
            xycoords='axes fraction', xytext=(0, 7), textcoords='offset points',
            fontsize=8.5, color=INK_MUTED, va='bottom')
plt.tight_layout()
plt.show()

Steps and distance are near-collinear (they are two views of the same pedometer
stream), and cardio points track active minutes almost exactly. The interesting,
*non*-redundant axis is mean speed: it separates "many slow steps" days from
"fewer, faster" days.

<a id="3"></a>
## 3. Intraday rhythm (15-minute buckets)

Each per-day CSV holds the same metrics split into 96 fifteen-minute buckets.
Loading all of them gives a minute-of-day view of routine.

In [ ]:
t0 = time.perf_counter()
frames = []
for path in sorted(DAILY_DIR.glob('[0-9][0-9][0-9][0-9]-[0-9][0-9]-[0-9][0-9].csv')):
    df = tidy_columns(pd.read_csv(path, encoding='utf-8'))
    df['date'] = pd.Timestamp(path.stem)
    frames.append(df)
intraday = pd.concat(frames, ignore_index=True)

# 'start' looks like 07:15:00.000+02:00 -> keep the local wall-clock part only.
intraday['minute_of_day'] = (
    intraday['start'].str.slice(0, 2).astype(int) * 60 + intraday['start'].str.slice(3, 5).astype(int)
)
intraday['hour'] = intraday['minute_of_day'] // 60
intraday['weekday'] = intraday['date'].dt.dayofweek
intraday['is_weekend'] = intraday['weekday'] >= 5
intraday['distance_km'] = intraday['distance_m'] / 1000

print(f'{len(intraday):,} buckets from {intraday.date.nunique():,} days '
      f'({time.perf_counter() - t0:.1f}s to load)')
intraday[['date', 'start', 'steps', 'distance_m', 'active_minutes', 'speed_mean_ms']].head()

In [ ]:
prof = (intraday.groupby(['is_weekend', 'minute_of_day'])['steps']
        .mean().unstack(0).rolling(3, center=True, min_periods=1).mean())

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(prof.index / 60, prof[False], color=SERIES[0], label='Weekday')
ax.plot(prof.index / 60, prof[True], color=SERIES[2], label='Weekend')
ax.set_xlim(0, 24)
ax.set_xticks(range(0, 25, 3), [f'{h:02d}:00' for h in range(0, 25, 3)])
ax.legend(loc='upper left', ncols=2)
finish(ax, 'When the steps happen',
       'mean steps per 15-minute bucket · lightly smoothed', ylabel='steps per bucket')
plt.show()

In [ ]:
hm = intraday.groupby(['weekday', 'hour'])['steps'].mean().unstack('hour')

fig, ax = plt.subplots(figsize=(11, 3.4))
im = ax.imshow(hm.values, aspect='auto', cmap=BLUES, interpolation='nearest')
ax.set_yticks(range(7), DAY_NAMES, fontsize=8)
ax.set_xticks(range(0, 24, 2), [f'{h:02d}' for h in range(0, 24, 2)], fontsize=8)
ax.grid(False)
for s in ax.spines.values():
    s.set_visible(False)
cb = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.015)
cb.outline.set_visible(False)
cb.ax.tick_params(labelsize=7.5)
ax.set_title('Average steps by hour and weekday', pad=22)
ax.annotate('darker = more steps · local time', xy=(0, 1), xycoords='axes fraction',
            xytext=(0, 7), textcoords='offset points', fontsize=8.5,
            color=INK_MUTED, va='bottom')
plt.tight_layout()
plt.show()

In [ ]:
# Where does the day's movement sit? Share of daily steps before noon.
day_tot = intraday.groupby('date')['steps'].sum()
morning = intraday[intraday.hour < 12].groupby('date')['steps'].sum()
share = (morning / day_tot).dropna()
share = share[day_tot.reindex(share.index) > 1000]

fig, ax = plt.subplots(figsize=(11, 3.4))
ax.hist(share * 100, bins=40, color=SERIES[0])
ax.axvline(share.median() * 100, color=SERIES[1], lw=2)
ax.annotate(f'median {share.median() * 100:.0f}%', xy=(share.median() * 100, 0),
            xytext=(6, 60), textcoords='offset points', fontsize=8.5, color=SERIES[1])
finish(ax, 'Share of the day\'s steps taken before noon',
       'days with more than 1,000 steps', ylabel='days', xlabel='% of daily steps')
plt.show()

<a id="4"></a>
## 4. Sessions

Every tracked workout has a JSON in `Todas las sesiones/` with a summary block.
The filename carries the *local* start time and UTC offset, which is what the
time-of-day analysis needs (`startTime` inside the file is UTC).

In [ ]:
FNAME_RE = re.compile(
    r'^(?P<d>\d{4}-\d{2}-\d{2})T(?P<h>\d{2})_(?P<mi>\d{2})_(?P<s>\d{2}(?:\.\d+)?)'
    r'(?P<off>[+-]\d{2})_(?P<offm>\d{2})_(?P<label>.+)$'
)

AGG_MAP = {
    'com.google.heart_minutes.summary': 'cardio_points',
    'com.google.calories.expended': 'calories',
    'com.google.distance.delta': 'distance_m',
    'com.google.speed.summary': 'speed_ms',
    'com.google.active_minutes': 'active_minutes',
    'com.google.step_count.delta': 'steps',
}

def parse_session(path):
    raw = json.loads(path.read_text(encoding='utf-8'))
    row = {
        'file': path.name,
        'activity': raw.get('fitnessActivity'),
        'start_utc': pd.Timestamp(raw['startTime']),
        'end_utc': pd.Timestamp(raw['endTime']),
        'duration_s': float(str(raw.get('duration', '0s')).rstrip('s')),
        'n_segments': len(raw.get('segment', [])),
    }
    m = FNAME_RE.match(path.stem)
    if m:
        row['start_local'] = pd.Timestamp(f"{m['d']} {m['h']}:{m['mi']}:{m['s']}")
    for a in raw.get('aggregate', []):
        key = AGG_MAP.get(a['metricName'])
        if key:
            row[key] = a.get('floatValue', a.get('intValue'))
    return row

sessions = pd.DataFrame(parse_session(p) for p in sorted(SESSIONS_DIR.glob('*.json')))
sessions['duration_min'] = sessions.duration_s / 60
sessions['distance_km'] = sessions.distance_m / 1000
sessions['date'] = sessions.start_local.dt.normalize()
sessions['month'] = sessions.start_local.dt.to_period('M').dt.to_timestamp()
sessions['hour'] = sessions.start_local.dt.hour + sessions.start_local.dt.minute / 60
sessions['weekday'] = sessions.start_local.dt.dayofweek
# Pace in min/km, only where a plausible distance exists.
sessions['pace_min_km'] = np.where(sessions.distance_km > 0.15,
                                   sessions.duration_min / sessions.distance_km, np.nan)

print(f'{len(sessions)} sessions, {sessions.start_local.min():%Y-%m-%d} -> {sessions.start_local.max():%Y-%m-%d}')
sessions.head()

In [ ]:
per_type = sessions.groupby('activity').agg(
    sessions=('file', 'size'),
    hours=('duration_min', lambda s: s.sum() / 60),
    total_km=('distance_km', 'sum'),
    median_min=('duration_min', 'median'),
    median_km=('distance_km', 'median'),
    median_pace=('pace_min_km', 'median'),
    cardio_points=('cardio_points', 'sum'),
).sort_values('sessions', ascending=False)
per_type.round(2)

Walking dominates: about 87% of sessions and the overwhelming majority of both
hours and kilometres. Two numbers in this table should not be read at face value:

- **Biking distance is not measurable here.** 53 hours of biking with 66 km
  recorded works out to about 1 km/h — a pedometer cannot count distance from a
  bicycle and there is no GPS in the export, so bike distance and bike pace are
  artefacts. Bike *duration* is still meaningful.
- **Median pace mixes commutes with exercise.** A 13 min/km walking median is a
  stroll-plus-stops number, not a walking speed; the pace chart below separates
  this by distance.

In [ ]:
counts = sessions.pivot_table(index='month', columns='activity', values='file',
                              aggfunc='size').fillna(0)
order = [a for a in ['walking', 'running', 'biking'] if a in counts.columns]

fig, ax = plt.subplots(figsize=(11, 4))
bottom = np.zeros(len(counts))
for act in order:
    ax.bar(counts.index, counts[act], bottom=bottom, width=22, label=act.capitalize(),
           color=ACT_COLOR[act], edgecolor=SURFACE, lw=1)
    bottom += counts[act].values
ax.legend(loc='upper left', ncols=3)
finish(ax, 'Tracked sessions per month', 'stacked by activity type', ylabel='sessions')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
bins_d = np.arange(0, 125, 5)
bins_k = np.arange(0, 16, 0.5)
for act in order:
    sub = sessions[sessions.activity == act]
    axes[0].hist(sub.duration_min.clip(upper=120), bins=bins_d, histtype='step', lw=2,
                 color=ACT_COLOR[act], label=act.capitalize())
    axes[1].hist(sub.distance_km.clip(upper=15), bins=bins_k, histtype='step', lw=2,
                 color=ACT_COLOR[act], label=act.capitalize())
finish(axes[0], 'Session duration', 'clipped at 120 min', ylabel='sessions', xlabel='minutes')
finish(axes[1], 'Session distance', 'clipped at 15 km', ylabel='sessions', xlabel='km')
axes[0].legend(loc='upper right')
plt.tight_layout()
plt.show()

In [ ]:
# Biking is excluded from every pace chart: step-derived distance is meaningless on a bike.
PACE_TYPES = [a for a in order if a != 'biking']

fig, ax = plt.subplots(figsize=(11, 4.6))
for act in PACE_TYPES:
    sub = sessions[(sessions.activity == act) & sessions.pace_min_km.between(2, 30)]
    ax.scatter(sub.distance_km, sub.pace_min_km, s=16, alpha=0.45,
               color=ACT_COLOR[act], edgecolors=SURFACE, linewidths=0.5, label=act.capitalize())
ax.set_xlim(0, sessions.distance_km.quantile(0.995))
ax.invert_yaxis()
ax.legend(loc='lower right', ncols=2)
finish(ax, 'Pace against distance', 'one dot per session · faster is higher · biking excluded',
       ylabel='pace (min/km)', xlabel='distance (km)', yfmt=False)
plt.show()

In [ ]:
pace_m = (sessions[sessions.pace_min_km.between(2, 30)]
          .pivot_table(index='month', columns='activity', values='pace_min_km', aggfunc='median'))

fig, ax = plt.subplots(figsize=(11, 4))
for act in PACE_TYPES:
    if act in pace_m:
        s = pace_m[act].dropna()
        ax.plot(s.index, s.values, color=ACT_COLOR[act], marker='o', ms=4, label=act.capitalize())
        ax.annotate(act.capitalize(), xy=(s.index[-1], s.iloc[-1]), xytext=(6, 0),
                    textcoords='offset points', fontsize=8.5, color=ACT_COLOR[act], va='center')
ax.invert_yaxis()
ax.legend(loc='lower left', ncols=2)
finish(ax, 'Median pace per month', 'lower line = faster · biking excluded',
       ylabel='min/km', yfmt=False)
plt.show()

In [ ]:
clock = sessions.pivot_table(index=sessions.start_local.dt.hour, columns='activity',
                             values='file', aggfunc='size').reindex(range(24)).fillna(0)

fig, ax = plt.subplots(figsize=(11, 3.8))
bottom = np.zeros(24)
for act in order:
    ax.bar(clock.index, clock[act], bottom=bottom, width=0.8, color=ACT_COLOR[act],
           label=act.capitalize(), edgecolor=SURFACE, lw=1)
    bottom += clock[act].values
ax.set_xticks(range(0, 24, 2), [f'{h:02d}:00' for h in range(0, 24, 2)])
ax.legend(loc='upper left', ncols=3)
finish(ax, 'What time sessions start', 'local time, all sessions', ylabel='sessions')
plt.show()

### Do the session files agree with the daily totals?

The daily CSV and the session JSONs are independent summaries of the same days,
so comparing them is a sanity check on both.

In [ ]:
sess_day = sessions.groupby('date').agg(sess_km=('distance_km', 'sum'),
                                        sess_min=('duration_min', 'sum'))
check = daily.set_index('date')[['distance_km']].join(sess_day, how='inner').dropna()

fig, ax = plt.subplots(figsize=(5.6, 5.4))
lim = max(check.distance_km.max(), check.sess_km.max()) * 1.05
ax.plot([0, lim], [0, lim], color=INK_MUTED, lw=1, ls=(0, (4, 3)))
ax.scatter(check.sess_km, check.distance_km, s=14, alpha=0.4, color=SERIES[0],
           edgecolors=SURFACE, linewidths=0.5)
ax.set_xlim(0, lim)
ax.set_ylim(0, lim)
finish(ax, 'Daily distance: sessions vs. daily CSV',
       'dashed line = perfect agreement', ylabel='daily CSV (km)', xlabel='sum of sessions (km)',
       yfmt=False)
plt.show()

share = (check.sess_km / check.distance_km).replace([np.inf, -np.inf], np.nan).dropna()
print(f'Sessions capture a median {share.median():.0%} of the daily distance '
      f'on days that have at least one session ({len(check)} days).')

<a id="5"></a>
## 5. TCX trackpoints

The TCX files are the highest-resolution part of the export: a distance value
every 20–60 seconds. Note what is **not** there — no `LatitudeDegrees` and no
`HeartRateBpm` in any file, so there is no GPS track and no heart rate to analyse.

One trap: a session is split into several `<Lap>` blocks (one per continuous
stretch of movement) and `DistanceMeters` **restarts at 0 in every lap**. Taking
the maximum over the whole file therefore loses everything but the longest lap —
about 30% of the distance on a typical walk. The parser below accumulates a lap
offset so the distance column is cumulative over the session.

In [ ]:
TCX_NS = {'t': 'http://www.garmin.com/xmlschemas/TrainingCenterDatabase/v2'}
_NS = '{' + TCX_NS['t'] + '}'
# Fully-qualified tags: Element.iter/find on a plain tag is much faster than an XPath.
TAG_LAP, TAG_TRACKPOINT = _NS + 'Lap', _NS + 'Trackpoint'
TAG_TIME, TAG_DISTANCE = _NS + 'Time', _NS + 'DistanceMeters'

def parse_tcx(path):
    '''Return (meta dict, trackpoint DataFrame) for one TCX file.

    DistanceMeters resets per lap, so laps are walked in order and each lap's
    total is added as an offset to build a session-cumulative distance.
    '''
    root = ET.parse(path).getroot()
    act = root.find('.//t:Activity', TCX_NS)
    rows, offset, n_laps = [], 0.0, 0
    for lap_i, lap in enumerate(root.iter(TAG_LAP)):
        n_laps += 1
        lap_max = 0.0
        for tp in lap.iter(TAG_TRACKPOINT):
            tm = tp.find(TAG_TIME)
            if tm is None:
                continue
            node = tp.find(TAG_DISTANCE)
            d = float(node.text) if node is not None and node.text else np.nan
            if d == d:                       # not NaN
                lap_max = max(lap_max, d)
            rows.append((tm.text, lap_i, d, offset + d))
        offset += lap_max
    tp_df = pd.DataFrame(rows, columns=['time', 'lap', 'lap_distance_m', 'distance_m'])
    if not tp_df.empty:
        tp_df['time'] = pd.to_datetime(tp_df['time'], format='mixed', utc=True)
        tp_df['elapsed_s'] = (tp_df.time - tp_df.time.iloc[0]).dt.total_seconds()
    meta = {
        'file': path.name,
        'sport': act.get('Sport') if act is not None else None,
        'n_laps': n_laps,
        'n_points': len(tp_df),
        'tcx_distance_m': offset,
        'tcx_duration_s': tp_df.elapsed_s.max() if not tp_df.empty else np.nan,
    }
    return meta, tp_df

t0 = time.perf_counter()
tcx = pd.DataFrame(parse_tcx(p)[0] for p in sorted(ACTIVITIES_DIR.glob('*.tcx')))
tcx['sampling_s'] = tcx.tcx_duration_s / tcx.n_points.clip(lower=1)
print(f'{len(tcx)} TCX files parsed in {time.perf_counter() - t0:.1f}s')
tcx.groupby('sport').agg(files=('file', 'size'), points=('n_points', 'sum'),
                         median_laps=('n_laps', 'median'),
                         median_points=('n_points', 'median'),
                         median_km=('tcx_distance_m', lambda s: s.median() / 1000),
                         median_sampling_s=('sampling_s', 'median')).round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.6))
ax.hist(tcx.sampling_s.clip(upper=120), bins=np.arange(0, 122, 2), color=SERIES[0])
finish(ax, 'Trackpoint sampling interval',
       'session duration / number of points · clipped at 120 s',
       ylabel='sessions', xlabel='seconds between points')
plt.show()

In [ ]:
# The three longest sessions, one curve each: distance against elapsed time.
longest = tcx.nlargest(3, 'tcx_distance_m')
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))
for i, row in enumerate(longest.itertuples()):
    _, tp = parse_tcx(ACTIVITIES_DIR / row.file)
    label = f'{row.file[:10]} · {row.sport.lower()} · {row.tcx_distance_m / 1000:.1f} km'
    ax1.plot(tp.elapsed_s / 60, tp.distance_m / 1000, color=SERIES[i], label=label)
    # Instantaneous speed, smoothed over ~5 samples. Pairs of points less than 5 s
    # apart are dropped: a metre of jitter over half a second reads as 100 km/h.
    dt = tp.elapsed_s.diff()
    ok = (dt >= 5) & (tp.lap.diff() == 0)
    spd = (tp.distance_m.diff() / dt).where(ok).rolling(5, min_periods=2, center=True).mean()
    ax2.plot(tp.elapsed_s / 60, spd * 3.6, color=SERIES[i], lw=1.6)
ax2.set_ylim(0, 12)
ax1.legend(loc='upper left', fontsize=8)
finish(ax1, 'Cumulative distance', 'the three longest tracked sessions',
       ylabel='km', xlabel='elapsed minutes', yfmt=False)
finish(ax2, 'Speed within the session', 'smoothed · sub-5-second gaps dropped',
       ylabel='km/h', xlabel='elapsed minutes', yfmt=False)
plt.tight_layout()
plt.show()

In [ ]:
# Cross-check: TCX distance vs. the session JSON aggregate for the same workout.
tcx_join = tcx.assign(key=tcx.file.str.slice(0, 19))
ses_join = sessions.assign(key=sessions.file.str.slice(0, 19))
both = tcx_join.merge(ses_join[['key', 'activity', 'distance_m', 'duration_s']],
                      on='key', suffixes=('_tcx', '_json'))
both['diff_pct'] = (both.tcx_distance_m - both.distance_m) / both.distance_m * 100
print(f'{len(both)} of {len(tcx)} TCX files matched to a session JSON')
print(both.diff_pct.describe(percentiles=[0.05, 0.5, 0.95]).round(2).to_string())

fig, ax = plt.subplots(figsize=(11, 3.6))
ax.hist(both.diff_pct.clip(-40, 40), bins=np.arange(-40, 41, 2), color=SERIES[0])
ax.axvline(0, color=INK_MUTED, lw=1, ls=(0, (4, 3)))
finish(ax, 'TCX distance minus session-JSON distance',
       'per session, as % of the JSON value · clipped at ±40%',
       ylabel='sessions', xlabel='% difference')
plt.show()

With the lap offsets applied the two sources line up: the median session now
differs by roughly +7% instead of −31%, and half of all sessions sit inside a few
percent. The remaining positive bias is expected — summing lap totals keeps
short wobbles that Google's own aggregation smooths away — and the long right
tail comes from short sessions where a few metres are a large percentage.

The general lesson: a plausible-looking column (a monotone cumulative distance)
can still be wrong, and it was the cross-check between two files describing the
same event that caught it.

<a id="6"></a>
## 6. Raw data streams (`Todos los datos/`)

These are the underlying Fit data streams: one JSON per data source, each a list
of data points with nanosecond timestamps. The files are large (about 500 MB in
total) but they are written one point per line, so they can be streamed instead
of loaded whole.

In [ ]:
def scan_stream(path, chunk=4 << 20):
    '''Cheap point count without parsing: files are one data point per line.'''
    n = 0
    with path.open('rb') as fh:
        while True:
            buf = fh.read(chunk)
            if not buf:
                break
            n += buf.count(b'"fitValue"')
    return n

catalog = []
for p in sorted(ALLDATA_DIR.glob('*.json')):
    name = p.stem
    kind, rest = (name.split('_', 1) + [''])[:2]
    catalog.append({
        'data_type': re.sub(r'^com\.google\.', '', rest.split('_com.google.android')[0]
                            .split('_cc.pacer')[0].split('_Xiaomi')[0]),
        'origin': 'derived' if kind == 'derived' else 'raw',
        'points': scan_stream(p),
        'size_mb': round(p.stat().st_size / 1e6, 1),
        'file': p.name,
    })
catalog = pd.DataFrame(catalog).sort_values('points', ascending=False).reset_index(drop=True)
catalog[['data_type', 'origin', 'points', 'size_mb']]

In [ ]:
empty = catalog[catalog.points == 0].groupby('data_type').size()
print('Data types present but EMPTY (file exists, nothing was ever recorded):')
for name, n in empty.items():
    print(f'  - {name}' + (f'  ({n} files)' if n > 1 else ''))

Worth calling out: **sleep, respiratory rate and paced-walking are empty**, and
weight/height have a single user-entered point each. This export is a
phone-pedometer dataset — no wearable, no heart-rate sensor, no sleep tracking.
So the questions this data can answer are about *ambulation over time*, not
about physiology or recovery.

In [ ]:
def iter_points(path):
    '''Yield data points one at a time (one JSON object per line).'''
    with path.open('r', encoding='utf-8') as fh:
        for line in fh:
            line = line.strip().rstrip(',')
            if line.startswith('{"fitValue"'):
                yield json.loads(line)

def _value(point):
    fv = point.get('fitValue') or [{}]
    v = fv[0].get('value', {})
    for k in ('intVal', 'fpVal', 'stringVal'):
        if k in v:
            return v[k]
    return np.nan

def load_stream(path, tz=TZ, limit=None):
    '''Load one data stream into a tidy DataFrame of start/end/value.'''
    rows = []
    for i, pt in enumerate(iter_points(path)):
        if limit and i >= limit:
            break
        rows.append((pt['startTimeNanos'], pt['endTimeNanos'], _value(pt)))
    df = pd.DataFrame(rows, columns=['start_ns', 'end_ns', 'value'])
    if df.empty:
        return df
    df['start'] = pd.to_datetime(df.start_ns, unit='ns', utc=True).dt.tz_convert(tz)
    df['end'] = pd.to_datetime(df.end_ns, unit='ns', utc=True).dt.tz_convert(tz)
    df['duration_s'] = (df.end_ns - df.start_ns) / 1e9
    return df.drop(columns=['start_ns', 'end_ns']).sort_values('start').reset_index(drop=True)

def find_stream(*keywords):
    for p in sorted(ALLDATA_DIR.glob('*.json')):
        n = p.name.lower()
        if all(k in n for k in keywords):
            return p
    raise FileNotFoundError(str(keywords))

steps_raw = load_stream(find_stream('step_count.delta', 'merge_step_deltas'))
print(f'{len(steps_raw):,} step-delta points, '
      f'{steps_raw.start.min():%Y-%m-%d} -> {steps_raw.start.max():%Y-%m-%d}')
steps_raw.head()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))
ax1.hist(steps_raw.duration_s.clip(upper=300), bins=60, color=SERIES[0])
finish(ax1, 'Length of a raw step-delta point', 'clipped at 300 s',
       ylabel='points', xlabel='seconds covered')
ax2.hist(steps_raw.value.clip(upper=200), bins=60, color=SERIES[0])
finish(ax2, 'Steps per raw point', 'clipped at 200', ylabel='points', xlabel='steps')
plt.tight_layout()
plt.show()

In [ ]:
# Zoom in on the single busiest day at raw resolution.
steps_raw['date'] = steps_raw.start.dt.normalize()
busiest = steps_raw.groupby('date')['value'].sum().idxmax()
day = steps_raw[steps_raw.date == busiest].copy()
day['minute'] = day.start.dt.hour + day.start.dt.minute / 60

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(day.minute, day.value, width=0.02, color=SERIES[0])
ax.set_xlim(0, 24)
ax.set_xticks(range(0, 25, 2), [f'{h:02d}:00' for h in range(0, 25, 2)])
finish(ax, f'Raw step deltas on the busiest day ({busiest:%A %d %b %Y})',
       f'{day.value.sum():,.0f} steps in {len(day):,} points', ylabel='steps per point')
plt.show()

In [ ]:
# Heart ("cardio") minutes: the one intensity signal the phone derives.
hm_raw = load_stream(find_stream('heart_minutes', 'merge_heart_minutes'))
hm_daily = hm_raw.set_index('start')['value'].resample('D').sum()

fig, ax = plt.subplots(figsize=(11, 3.8))
ax.plot(hm_daily.index, hm_daily.values, color=SERIES[0], lw=0.7, alpha=0.3)
ax.plot(hm_daily.index, hm_daily.rolling(30, min_periods=10).mean(), color=SERIES[0], lw=2.2)
finish(ax, 'Cardio points per day (raw stream)',
       f'{len(hm_raw):,} points · 30-day average in bold', ylabel='cardio points')
plt.show()

print(hm_daily.describe().round(1).to_string())

In [ ]:
# Activity segments: how the phone classified time, by minutes per activity code.
ACTIVITY_CODES = {
    0: 'in vehicle', 1: 'biking', 2: 'on foot', 3: 'still', 4: 'unknown', 5: 'tilting',
    7: 'walking', 8: 'running', 72: 'sleeping', 108: 'elliptical', 109: 'other',
}
seg = load_stream(find_stream('activity.segment', 'merge_activity_segments.json'))
seg['activity'] = seg.value.map(ACTIVITY_CODES).fillna(seg.value.astype(str))
seg_summary = (seg.groupby('activity')
               .agg(segments=('value', 'size'), hours=('duration_s', lambda s: s.sum() / 3600))
               .sort_values('hours', ascending=False))
seg_summary.round(1)

In [ ]:
top = seg_summary.head(6).iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 3.4))
bars = ax.barh(top.index, top.hours, color=SERIES[0], height=0.66)
for bar, v in zip(bars, top.hours):
    ax.text(bar.get_width() + top.hours.max() * 0.012, bar.get_y() + bar.get_height() / 2,
            f'{v:,.0f} h', va='center', fontsize=8.5, color=INK_SOFT)
ax.grid(axis='y', visible=False)
ax.grid(axis='x', visible=True)
ax.set_xlim(0, top.hours.max() * 1.15)
ax.set_title('Hours per classified activity segment', pad=22)
ax.annotate('derived activity segments across the whole export', xy=(0, 1),
            xycoords='axes fraction', xytext=(0, 7), textcoords='offset points',
            fontsize=8.5, color=INK_MUTED, va='bottom')
plt.tight_layout()
plt.show()

In [ ]:
# Single-point streams: weight and height as entered by hand.
for keys in [('weight', 'merge_weight'), ('height', 'merge_height')]:
    df = load_stream(find_stream(*keys))
    label = keys[0]
    if df.empty:
        print(f'{label}: no data')
    else:
        print(f'{label}: {len(df)} point(s)')
        print(df[['start', 'value']].to_string(index=False))

<a id="7"></a>
## 7. Takeaways

Run the cell below to print the headline numbers for this export, then read the notes.

In [ ]:
span_days = (daily.date.max() - daily.date.min()).days + 1
print(f'Coverage        : {daily.date.min():%d %b %Y} -> {daily.date.max():%d %b %Y} '
      f'({span_days} days, {len(daily)} with data)')
print(f'Total steps     : {daily.steps.sum():,.0f}')
print(f'Total distance  : {daily.distance_km.sum():,.0f} km')
print(f'Median day      : {daily.steps.median():,.0f} steps / {daily.distance_km.median():.1f} km')
print(f'Best day        : {daily.steps.max():,.0f} steps on '
      f'{daily.loc[daily.steps.idxmax(), "date"]:%d %b %Y}')
print(f'10k-step days   : {(daily.steps >= 10000).mean():.0%}')
print(f'Tracked sessions: {len(sessions):,} '
      f'({sessions.duration_min.sum() / 60:,.0f} h, {sessions.distance_km.sum():,.0f} km)')
print(f'Raw data points : {catalog.points.sum():,} across {len(catalog)} streams')

**What the data supports**

- Roughly three years of continuous pedometer coverage, with essentially no
  missing days — unusually clean for a Takeout export.
- Steps, distance and speed are complete and mutually consistent; the daily CSV,
  the session JSONs and the TCX files agree with each other where they overlap.
- Two distinct behavioural signals are visible: a weekday/weekend split, and a
  strong time-of-day routine.

**What it does not support**

- No sleep, no heart rate, no respiratory rate — those streams exist in the export
  but are empty. Anything about recovery or intensity beyond derived "cardio
  points" is out of reach.
- Calories are only present on ~60% of days and come partly from a different app,
  so trends in calories mostly track *which app was installed*, not energy expenditure.
- Weight is a single hand-entered value; there is no body-composition series.
- The TCX files contain no GPS coordinates, so no route or elevation analysis is possible.
- Biking distance and pace are artefacts of a step-counter running in a pocket on
  a bike; only biking *duration* is usable.

**Natural next steps**

- Model the weekday/weekend and seasonal components of daily steps and look at
  the residuals — the unusual days are where the stories are.
- Split sessions into "commute-shaped" and "exercise-shaped" using duration,
  pace and start hour, then track the mix over time.
- Compare the 2023–24 phone against the 2025+ phone on the same metric: the
  device change is a plausible source of step-count bias worth quantifying before
  reading any long-term trend as behaviour.